# Stage 1 — Non-Instruction Fine-Tuning (Domain Pretraining)
### HR Policy Assistant · powered by Unsloth · pushed to Hugging Face

**Notebook 1 of 4.** The four notebooks chain through your Hugging Face account:

```text
Stage 1 (this)  base Qwen2.5-1.5B + raw HR policy text  →  push  <you>/hr-policy-assistant-stage1
Stage 2 (SFT)   load stage1 from Hub + HR Q&A pairs     →  push  <you>/hr-policy-assistant-stage2
Stage 3 (DPO)   load stage2 from Hub + preference pairs →  push  <you>/hr-policy-assistant-final
Stage 4 (Eval)  pull all three and compare side by side
```

### What this stage does, in plain words

We show the model **raw HR policy text** and ask it to do one thing: **predict the
next word**. No questions, no answers — just text.

Why bother? Because the base model has never read your HR policy. It doesn't know
that "casual leave" is a thing, or that policies are written in a particular clipped,
formal style. This stage teaches it the **vocabulary and voice of the domain**.

It does **not** teach it to answer questions — after this stage it will happily
*continue* a sentence but won't behave like an assistant. That's Stage 2's job.

> **Runtime:** Colab → Runtime → Change runtime type → **T4 GPU**.

## 1. Install libraries

**Unsloth** is the engine here: it rewrites the training kernels so LoRA
fine-tuning runs roughly 2× faster with ~50% less VRAM, using the same
Hugging Face API you already know.

In [1]:
# ============================================================
# Step 1. Install libraries  (~2-3 min)
# ============================================================
!pip install -q unsloth
!pip install -q transformers trl datasets peft bitsandbytes accelerate sentencepiece protobuf huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 12.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 758.1 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 110.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 20.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 2.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Imports and GPU check

In [2]:
# ============================================================
# Step 2. Imports + GPU check
# ============================================================
import torch, json, gc
from unsloth import FastLanguageModel      # Unsloth's fast model loader
from trl import SFTTrainer, SFTConfig      # Supervised fine-tuning trainer
from datasets import Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: No GPU. Training will be extremely slow. Enable a T4.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## Hugging Face login

Every stage pushes its merged model to the Hub, and the next stage pulls it back
down. So we log in once, here, at the top.

**Never paste a raw token into a notebook cell.** A token in a saved `.ipynb` is
a leaked credential — anyone with the file can push or delete on your account.
Use a **Colab secret** instead:

> Click the **key icon (🔑)** in the Colab left sidebar → **Add new secret** →
> Name: `HF_TOKEN`, Value: your token from
> https://huggingface.co/settings/tokens (needs **write** access) → toggle
> **Notebook access** on.

The cell below reads that secret automatically, and falls back to an interactive
prompt if it isn't set.

In [3]:
# ============================================================
# Hugging Face login
# ============================================================
from huggingface_hub import login, whoami

try:
    # Colab secret named HF_TOKEN (recommended).
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    # Fallback: paste the token when prompted.
    login()

HF_USERNAME = whoami()["name"]
print("Logged in as:", HF_USERNAME)

Logged in as: Mohan143


## 3. Configuration

All knobs in one place. The two concept blocks below explain every value.

In [4]:
# ============================================================
# Step 3. Configuration
# ============================================================
# --- Model ---
model_name      = "unsloth/Qwen2.5-1.5B-bnb-4bit"  # base model, pre-quantized to 4-bit
max_seq_length  = 512      # max tokens per training sample (HR paragraphs are short)
dtype           = None     # None = auto-detect best dtype for this GPU
load_in_4bit    = True     # 4-bit quantization -> fits a free T4 (see concept below)

# --- LoRA ---
lora_rank      = 16
lora_alpha     = 32        # = 2 x rank (common heuristic)
lora_dropout   = 0         # Unsloth's fast path is optimized for 0
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

# --- Training ---
learning_rate               = 2e-4   # higher: we want a real domain shift
num_train_epochs            = 3
per_device_train_batch_size = 1
gradient_accumulation_steps = 8      # effective batch = 1 x 8 = 8
warmup_steps                = 50
logging_steps               = 20
save_steps                  = 100
seed                        = 42

# --- Hub repo for this stage's output ---
STAGE1_REPO = f"{HF_USERNAME}/hr-policy-assistant-stage1"
PRIVATE     = True

print("Config ready.")
print(f"Base model : {model_name}")
print(f"LoRA       : rank={lora_rank}, alpha={lora_alpha}")
print(f"Training   : lr={learning_rate}, epochs={num_train_epochs}, eff.batch={per_device_train_batch_size*gradient_accumulation_steps}")
print(f"Will push to: {STAGE1_REPO}")

Config ready.
Base model : unsloth/Qwen2.5-1.5B-bnb-4bit
LoRA       : rank=16, alpha=32
Training   : lr=0.0002, epochs=3, eff.batch=8
Will push to: Mohan143/hr-policy-assistant-stage1


### Concept: quantization (what `load_in_4bit` actually does)

A model's weights are numbers. **Precision** is how many bits we use per number.

| Precision | Bits/weight | Memory for a 1.5B model | Note |
|---|---|---|---|
| float32 (full) | 32 | ~6.0 GB | Original training precision |
| float16 / bfloat16 (half) | 16 | ~3.0 GB | Standard for inference |
| **4-bit (NF4)** | **4** | **~0.9 GB** | What we use — fits a free T4 easily |

**Quantization** = storing those weights in fewer bits. It's like saving a photo
as a smaller JPEG: slightly less detail, dramatically less space.

- **NF4** ("4-bit NormalFloat") is a 4-bit format designed for the bell-curve
  shape that neural-network weights actually follow, so it loses less accuracy
  than naive 4-bit rounding.
- The quantized weights stay **frozen**. Maths still happens in 16-bit, so
  quality loss is small.
- **QLoRA** = *quantized base model* + *LoRA adapter trained on top*. That
  combination is what makes fine-tuning a 1.5B model possible on a free GPU.

Trade-off: 4-bit saves memory but is slightly slower per step and marginally
less precise. For fine-tuning on a T4, it's the right call.

### Concept: LoRA (why we train ~1% of the model)

Full fine-tuning updates **every** weight — expensive, and you must store a whole
new model each time.

**LoRA (Low-Rank Adaptation)** freezes the base model and injects a small pair of
trainable matrices into chosen layers. We train only those.

```text
Output = FrozenBaseWeights(x)  +  (B · A)(x) · (alpha / r)
                                  └─ tiny, trainable ─┘
```

| Argument | Meaning | Why this value |
|---|---|---|
| `r` (rank) | Size/capacity of the adapter | 16 — enough for a domain shift, still tiny |
| `lora_alpha` | Scales the adapter's effect (`alpha/r`) | 32 = 2×rank, a common, stable default |
| `lora_dropout` | Randomly drops adapter units to fight overfitting | 0 — Unsloth's optimized path is fastest at 0 |
| `target_modules` | Which layers get an adapter | All attention (`q,k,v,o_proj`) + MLP (`gate,up,down_proj`) = best quality |
| `use_gradient_checkpointing` | Recompute activations instead of storing them | `"unsloth"` — saves the most VRAM, allows longer sequences |
| `bias` | Whether to train bias terms | `"none"` — standard, keeps adapter small |

Result: trainable parameters drop from ~1.5B to ~18M (about 1%).

## 4. Load the HR policy corpus

The data is **plain text** — no instructions, no Q&A. We split it into paragraphs
and keep only ones longer than 50 characters (short lines are usually headings or
noise, which teach the model nothing).

Point `DATA_PATH` at your own file, or let the fallback sample run so the
notebook works end-to-end out of the box.

In [7]:
# import os, shutil
# from google.colab import files

# DEST = "/content/data"          # your target folder
# os.makedirs(DEST, exist_ok=True)

# uploaded = files.upload()

# for fn in uploaded.keys():
#     shutil.move(fn, os.path.join(DEST, fn))
#     print(f'Uploaded "{fn}" ({len(uploaded[fn])} bytes) -> {DEST}/{fn}')

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [8]:
# ============================================================
# Step 4. Load raw HR policy text -> paragraphs
# ============================================================
DATA_PATH = "/content/drive/MyDrive/AgenticAI/HR-Finetuning/non_instruction_data.txt"

try:
    with open(DATA_PATH, "r", encoding="utf-8") as f:
        hr_text = f.read()
    print(f"Loaded {len(hr_text):,} characters from file.")
except FileNotFoundError:
    print("File not found -> using a small embedded sample (replace with your real corpus).")
    hr_text = """All full-time employees are entitled to 12 days of casual leave per calendar year. Casual leave cannot be carried forward to the next year and must be utilized within the same calendar year. Unused casual leave will lapse automatically on December 31st of each year.

Sick leave is granted at 10 days per year for all confirmed employees. Employees must notify their immediate supervisor within the first two hours of their scheduled shift if they are unable to attend work due to illness. Medical documentation is required for sick leave exceeding three consecutive days.

The standard working hours for all employees are from 9:00 AM to 6:00 PM, Monday through Friday. A lunch break of one hour is provided between 1:00 PM and 2:00 PM. Employees may choose flexible timing between 8:00 AM to 10:00 AM start time with corresponding end time adjustments, subject to manager approval.

The hybrid work policy allows eligible employees to work remotely for up to 2 days per week. Wednesday is designated as a mandatory in-office day for all employees to facilitate team collaboration and meetings. Teams may designate additional mandatory in-office days based on project requirements.

The notice period for confirmed employees is 60 days for roles up to senior manager level and 90 days for roles at director level and above. Notice period must be served in full unless buyout is approved by the department head and HR."""

# Split into paragraphs; drop anything too short to be useful.
paragraphs = [p.strip() for p in hr_text.split("\n\n") if len(p.strip()) > 50]

print(f"Paragraphs kept   : {len(paragraphs)}")
print(f"Avg length (chars): {sum(len(p) for p in paragraphs)//len(paragraphs)}")
print(f"\nSample:\n{paragraphs[0][:220]}...")

Loaded 93,952 characters from file.
Paragraphs kept   : 304
Avg length (chars): 307

Sample:
All full-time employees are entitled to 12 days of casual leave per calendar year. Casual leave cannot be carried forward to the next year and must be utilized within the same calendar year. Unused casual leave will laps...


In [9]:
# ============================================================
# Step 4b. Build a Hugging Face Dataset (one "text" column)
# ============================================================
dataset = Dataset.from_dict({"text": paragraphs})
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 304
})


## 5. Load the base model and attach LoRA

Unsloth's `FastLanguageModel.from_pretrained` loads the 4-bit base;
`get_peft_model` bolts on the trainable LoRA adapter. Watch the trainable-parameter
count in the output — that's LoRA's whole point.

In [10]:
# ============================================================
# Step 5. Load 4-bit base model + apply LoRA
# ============================================================
print("Loading base model... (1-2 min)")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = model_name,
    max_seq_length = max_seq_length,
    dtype          = dtype,
    load_in_4bit   = load_in_4bit,
)

print("Applying LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r                         = lora_rank,
    target_modules            = target_modules,
    lora_alpha                = lora_alpha,
    lora_dropout              = lora_dropout,
    bias                      = "none",
    use_gradient_checkpointing = "unsloth",   # max VRAM savings
    random_state              = seed,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"\nTrainable params: {trainable:,}")
print(f"Total params    : {total:,}")
print(f"Training only   : {100*trainable/total:.2f}% of the model")

Loading base model... (1-2 min)
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Applying LoRA adapters...


Unsloth 2026.7.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.



Trainable params: 18,464,768
Total params    : 907,081,216
Training only   : 2.04% of the model


### Concept: the training arguments

One Stage-1-specific argument matters most: **`packing=True`**. Our paragraphs are short; without packing each one is padded out to 512 tokens and we'd waste most of the compute on padding. Packing concatenates paragraphs back-to-back and slices them into full 512-token blocks, so nearly every token is real text. This is ideal for raw-text pretraining (in Stage 2 we turn it **off**, because instruction examples must stay separate).

| Argument | What it does | Why this value |
|---|---|---|
| `learning_rate` | Step size for each weight update | **2e-4** — relatively high. This stage makes a big shift (general model → HR domain), so we want meaningful steps. |
| `num_train_epochs` | How many full passes over the data | 3 — enough to learn, few enough to avoid memorizing |
| `per_device_train_batch_size` | Examples per GPU step | 1 — keeps VRAM low on a T4 |
| `gradient_accumulation_steps` | Accumulate grads over N steps, then update once | 8 → **effective batch = 1×8 = 8** (big-batch stability, small-batch memory) |
| `warmup_steps` | Slowly ramp the LR up at the start | Prevents a large, destabilizing first update |
| `optim="adamw_8bit"` | The optimizer, in 8-bit | Adam stores 2 extra numbers per weight; 8-bit cuts that memory ~4× |
| `weight_decay=0.01` | Mild penalty on large weights | Standard regularization; reduces overfitting |
| `lr_scheduler_type="linear"` | LR decays linearly to 0 | Big steps early, fine-tuning steps late |
| `fp16` / `bf16` | Mixed-precision maths | Auto-picked: `bf16` on newer GPUs (more stable), `fp16` on T4 |
| `seed` | Fixes randomness | 42 — makes the run reproducible |
| `logging_steps` | How often to print the loss | So you can watch it fall |

In [11]:
# ============================================================
# Step 6. SFTTrainer for Stage 1 (raw text, packing ON)
# ============================================================
trainer = SFTTrainer(
    model             = model,
    tokenizer         = tokenizer,
    train_dataset     = dataset,
    dataset_text_field = "text",
    max_seq_length    = max_seq_length,
    dataset_num_proc  = 2,
    packing           = True,     # <-- pack short paragraphs into full blocks
    args = SFTConfig(
        per_device_train_batch_size = per_device_train_batch_size,
        gradient_accumulation_steps = gradient_accumulation_steps,
        warmup_steps        = warmup_steps,
        num_train_epochs    = num_train_epochs,
        learning_rate       = learning_rate,
        fp16                = not torch.cuda.is_bf16_supported(),
        bf16                = torch.cuda.is_bf16_supported(),
        logging_steps       = logging_steps,
        save_steps          = save_steps,
        optim               = "adamw_8bit",
        weight_decay        = 0.01,
        lr_scheduler_type   = "linear",
        seed                = seed,
        output_dir          = "outputs/stage1_non_instruction",
        report_to           = "none",
    ),
)
print("SFTTrainer ready | packing=True | samples:", len(dataset))

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/304 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
SFTTrainer ready | packing=True | samples: 304


## 7. Train

Watch the **loss** column fall. There's a full explanation of what that number
means at the bottom of this notebook.

In [12]:
# ============================================================
# Step 7. Train  (5-10 min on a T4)
# ============================================================
print("Starting Stage 1 training...")
stats = trainer.train()

print("\nTraining complete.")
print(f"Runtime    : {stats.metrics['train_runtime']:.0f} s")
print(f"Samples/sec: {stats.metrics['train_samples_per_second']:.2f}")
print(f"Final loss : {stats.metrics['train_loss']:.4f}")
print(f"Perplexity : {torch.exp(torch.tensor(stats.metrics['train_loss'])):.2f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting Stage 1 training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 304 | Num Epochs = 3 | Total steps = 114
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
20,2.748032
40,2.212239
60,1.850124
80,1.638890
100,1.064994


Unsloth: Restored added_tokens_decoder metadata in outputs/stage1_non_instruction/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/stage1_non_instruction/checkpoint-114/tokenizer_config.json.



Training complete.
Runtime    : 465 s
Samples/sec: 1.96
Final loss : 1.7983
Perplexity : 6.04


## 8. Save and push the merged model

**Why merge?** Right now the model is "base + adapter" — two pieces. Stage 2 needs
a single, self-contained model to build on. `save_pretrained_merged` with
`merged_16bit` folds the LoRA weights into the base and writes one standalone
16-bit model.

We push that merged model to the Hub so **Stage 2 can load it from anywhere** —
no local files, no Drive dependency.

In [13]:
# ============================================================
# Step 8. Save adapter, merge, and push to the Hub
# ============================================================
# 8a. The adapter alone (small, ~100 MB) - useful as a backup.
model.save_pretrained("stage1_lora_adapter")
tokenizer.save_pretrained("stage1_lora_adapter")
print("Adapter saved -> stage1_lora_adapter/")

# 8b. Merged standalone model (base weights + adapter folded in).
print("\nMerging LoRA into base weights...")
model.save_pretrained_merged("stage1_merged_model", tokenizer, save_method="merged_16bit")
print("Merged model saved -> stage1_merged_model/")

# 8c. Push the MERGED model to the Hub (this is what Stage 2 loads).
print(f"\nPushing to {STAGE1_REPO} ... (5-10 min)")
model.push_to_hub_merged(STAGE1_REPO, tokenizer, save_method="merged_16bit", private=PRIVATE)
print(f"Done -> https://huggingface.co/{STAGE1_REPO}")

Unsloth: Restored added_tokens_decoder metadata in stage1_lora_adapter/tokenizer_config.json.


Adapter saved -> stage1_lora_adapter/

Merging LoRA into base weights...


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in stage1_merged_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-stage1/model.safetensors:   1%|          | 30.3MB / 3.09GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:22<00:00, 82.03s/it]


Unsloth: Merge process complete. Saved to `/content/Mohan143/hr-policy-assistant-stage1`
Done -> https://huggingface.co/Mohan143/hr-policy-assistant-stage1


## 9. Sanity check — text continuation

This stage is **not** a chatbot. So we don't ask it questions; we give it the
**start of a sentence** and let it continue. If it continues in HR-policy voice
with plausible HR terms, Stage 1 worked.

In [14]:
# ============================================================
# Step 9. Test: does it continue text in HR-policy style?
# ============================================================
FastLanguageModel.for_inference(model)   # ~2x faster generation

prompts = [
    "Employees are entitled to",
    "The company provides health insurance",
    "Performance reviews are conducted",
]

print("=" * 80)
print("STAGE 1 TEST - Text Continuation (not Q&A)")
print("=" * 80)
for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=90, temperature=0.7,
                             top_p=0.9, do_sample=True,
                             pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nPrompt   : {prompt}")
    print(f"Continues: {text[len(prompt):][:280]}")
    print("-" * 80)


Both `max_new_tokens` (=90) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


STAGE 1 TEST - Text Continuation (not Q&A)


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=90) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt   : Employees are entitled to
Continues:  10 days of earned time off per year for their birthday month. This earned time off does not accumulate and must be used in the same calendar year. Unused earned time off does not carry forward to the next year. Employees must apply for earned time off at least 7 days in advance 
--------------------------------------------------------------------------------


Both `max_new_tokens` (=90) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Prompt   : The company provides health insurance
Continues:  coverage through its partnership with an approved insurance provider. Employees can opt for add-on coverage for critical illness, dental, or maternity expenses through the insurance portal. Premiums are deducted directly from the employee's salary. The company covers 50 percent 
--------------------------------------------------------------------------------

Prompt   : Performance reviews are conducted
Continues:  quarterly by the reporting manager and are scored on a 1-4 scale based on performance, growth potential, and engagement. The review process ensures transparency, constructive feedback, and development action planning. Review outcomes inform compensation adjustments and engagemen
--------------------------------------------------------------------------------


### Concept: quantization (what `load_in_4bit` actually does)

#### The basic idea

A model's weights are just numbers. **Precision** is how many bits we spend
storing each one. Fewer bits = less memory, slightly less detail.

| Precision | Bits | Bytes/weight | Memory for a 1.5B model | Note |
|---|---|---|---|---|
| float32 (full) | 32 | 4.0 | ~6.0 GB | Original training precision |
| float16 / bfloat16 (half) | 16 | 2.0 | ~3.0 GB | Standard for inference |
| int8 | 8 | 1.0 | ~1.5 GB | Middle ground |
| **4-bit (NF4)** | **4** | **0.5** | **~0.9 GB** | What we use — fits a free T4 easily |

The maths is simply: `params × bytes_per_weight`. So 1.5B × 0.5 bytes ≈ 0.75 GB,
plus a little overhead for the quantization bookkeeping → ~0.9 GB in practice.

**Quantization** = storing those weights in fewer bits. It's like saving a photo
as a smaller JPEG: slightly less detail, dramatically less space.

#### Why we care (the T4 budget)

A free Colab T4 has **~15 GB of VRAM**, and the model weights are only one
tenant. You also pay for:

| What | Roughly |
|---|---|
| Model weights | 0.9 GB (4-bit) vs 3.0 GB (16-bit) |
| LoRA adapter + its gradients | small (~100 MB) |
| Optimizer state (AdamW keeps 2 extra numbers per *trainable* weight) | small, because LoRA means few trainable weights |
| **Activations** (the intermediate values of every layer) | often the biggest chunk — grows with `max_seq_length` and batch size |
| CUDA/framework overhead | ~1 GB |

Loading in 4-bit frees ~2 GB, and that headroom is what buys you a longer
`max_seq_length` and a workable batch size. This is also why we set
`use_gradient_checkpointing="unsloth"` — it attacks the *activation* half of the
bill.

#### How 4 bits can possibly be enough

4 bits = **16 distinct values**. Squeezing billions of weights into 16 buckets
sounds hopeless — but two tricks make it work.

**1. NF4 ("4-bit NormalFloat") matches the shape of the data.**
Neural network weights aren't random; they cluster tightly around zero in a
bell curve. Naive 4-bit spreads its 16 values evenly, wasting most of them on
extreme values that barely occur. NF4 places its 16 values so each one carries
roughly equal probability mass — dense near zero, sparse at the tails. It's
information-theoretically optimal for normally distributed data, which is exactly
what weights are.

```text
Naive INT4:  | . . . . . . . . |     evenly spaced, most buckets wasted
NF4:         |....|..|.|.|..|....|   dense where the weights actually live
```

**2. Block-wise quantization stops outliers ruining everything.**
Instead of one scale factor for the whole tensor, weights are quantized in small
**blocks** (64 weights each), every block with its own scale. One freak outlier
can only distort its own block, not the entire layer.

`bnb_4bit_use_double_quant=True` takes this one step further: it quantizes the
*scale factors themselves*, saving roughly another 0.4 bits per weight. Small,
free, no real downside.

#### Storage is 4-bit, but the maths isn't

This is the part that confuses people. The weights **sit in memory as 4-bit**,
but nothing is actually computed in 4-bit:

```text
4-bit weight in VRAM  →  dequantize to 16-bit  →  do the matmul in 16-bit  →  discard
```

That's what `bnb_4bit_compute_dtype=torch.float16` means — the *compute* dtype,
separate from the *storage* dtype. So you get 16-bit-quality arithmetic at 4-bit
memory cost, paying only a small dequantization overhead per operation.

#### What stays frozen, and why that's the whole trick

The quantized base weights are **frozen** — never updated. You can't cleanly
backprop into a 4-bit number anyway.

So what trains? The **LoRA adapter**, which lives in full precision alongside the
frozen 4-bit base:

```text
Output = FrozenBase_4bit(x)  +  LoRA_16bit(x)
         └─ quantized, frozen ─┘   └─ tiny, trainable ─┘
```

**QLoRA = quantized base + LoRA adapter on top.** That combination is what makes
fine-tuning a 1.5B model possible on a free GPU. The QLoRA authors showed this
matches full 16-bit fine-tuning quality — the quantization error lands in the
frozen part, and the trainable adapter simply learns around it.

#### The arguments, decoded

| Argument | What it does |
|---|---|
| `load_in_4bit=True` | Turn on 4-bit storage |
| `bnb_4bit_quant_type="nf4"` | Use NF4 (vs `"fp4"`, the older/less accurate option) |
| `bnb_4bit_compute_dtype=torch.float16` | Dequantize to fp16 for the actual maths |
| `bnb_4bit_use_double_quant=True` | Also quantize the scale factors (~0.4 extra bits saved) |
| `dtype=None` (Unsloth) | Auto-pick the best compute dtype for this GPU |

Unsloth's `unsloth/Qwen2.5-1.5B-bnb-4bit` is **pre-quantized** — the 4-bit
version is already on the Hub, so you skip both the 3 GB download and the
quantize-on-load step.

#### The trade-offs (when *not* to use it)

| | 4-bit | 16-bit |
|---|---|---|
| Memory | ~0.9 GB | ~3.0 GB |
| Speed per step | **slower** (dequantization overhead) | faster |
| Precision | tiny loss | none |
| Fine-tuning on a free T4 | ✅ the only way | ❌ won't fit comfortably |
| High-throughput serving | ❌ wasteful | ✅ better |

The counterintuitive bit: **4-bit is not faster.** It trades speed for memory.
You use it because the alternative isn't "slower" — it's "doesn't run at all".

This is also why we save each stage with `save_method="merged_16bit"` rather than
4-bit: we quantize to *train*, then merge back to 16-bit so the next stage (and
eventually production) gets a clean, full-quality model to build on.

**Rule of thumb:** quantize to fine-tune on limited hardware. Save and serve in
16-bit when you can afford the memory.

## Understanding the loss function (read this — it's the whole game)

The number printed during training is the **cross-entropy loss** of next-token
prediction. Everything in Stages 1 and 2 is optimizing exactly this.

### What the model actually does

At every position, the model outputs a **probability for every token in the
vocabulary** — "what comes next?" The loss asks one question:

> **What probability did the model assign to the token that actually came next?**

```text
Loss for one token = -log( probability assigned to the correct token )
```

The minus-log is the key. Look at how it behaves:

| Model's probability for the correct token | Loss | Meaning |
|---|---|---|
| 1.00 (certain, correct) | 0.00 | Perfect — no penalty |
| 0.50 | 0.69 | Unsure |
| 0.10 | 2.30 | Mostly wrong |
| 0.01 | 4.61 | Confidently wrong — heavily punished |

So the loss is **small when the model is confidently right**, and **explodes when
the model is confidently wrong**. The total loss is this value averaged over
every token in the batch. Training nudges the weights to push the correct token's
probability up.

### Reading your numbers

- **Loss falling** → the model is assigning higher probability to real HR policy
  text. It's learning.
- **Loss flat from step 1** → learning rate too low, or data/format broken.
- **Loss spiking / NaN** → learning rate too high.
- **Training loss falls but validation loss rises** → **overfitting**: it's
  memorizing your examples instead of learning patterns. Use fewer epochs or
  more data.

### A useful intuition: perplexity

```python
perplexity = exp(loss)
```

Perplexity is "how many tokens is the model effectively torn between?"
Loss 2.30 → perplexity 10 → it's about as confused as if it were guessing among
10 options. **Lower is better.** A loss around 1.0–2.0 on domain text is
typically healthy for this kind of small-model fine-tune.

> ⚠️ **Low loss ≠ good model.** Loss only measures "did it predict the next token
> of *my training text*". It does not measure whether an answer is *correct*.
> That's why we test with real questions — and why Stage 3 exists.

### In Stage 1 specifically

The model is graded on **every token** of the HR text — there's no
"question" or "answer", it's all just text to predict. So the loss here measures
one thing: *how well has the model internalized the language of your HR policy?*

---

## Stage 1 complete

Your domain-adapted model is on the Hub at `STAGE1_REPO`.

**Next:** open **`Stage2_Instruction_SFT.ipynb`**. It pulls this model down and
teaches it to actually answer questions.